In [0]:
pwd

'/Workspace/Repos/divya.dhaipullay@zeussolutionsinc.com/automate_ddr'

In [0]:
%run ./init

In [0]:
import os
import logging
from pdf2image import convert_from_path
import pytesseract

# Configure logging (as in your notebook)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("PDFConverter")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(ch)

class PDFToImageConverter:
    """
    Converts all PDFs in `pdf_dir` into per‑page PNGs under
    `output_dir/<pdf_basename>/page_<n>.png`, and any page containing the
    term "closed deal sheet" into `/closed_deal_sheets/<pdf_basename>_page_<n>.png`.
    """

    def __init__(self, pdf_dir: str, output_dir: str, dpi: int = 300):
        self.pdf_dir = pdf_dir
        self.output_dir = output_dir
        self.dpi = dpi

        os.makedirs(self.output_dir, exist_ok=True)
        # Folder for closed deal sheets
        self.closed_dir = os.path.join(self.output_dir, "closed_deal_sheets")
        os.makedirs(self.closed_dir, exist_ok=True)

    def convert_all(self):
        pdf_files = [
            f for f in os.listdir(self.pdf_dir)
            if f.lower().endswith(".pdf")
        ]
        logger.info(f"Found {len(pdf_files)} PDFs in {self.pdf_dir}")

        for pdf in pdf_files:
            base = os.path.splitext(pdf)[0]
            folder = os.path.join(self.output_dir, base)
            os.makedirs(folder, exist_ok=True)

            logger.info(f"Converting {pdf} → images in {folder}")
            pdf_path = os.path.join(self.pdf_dir, pdf)
            pages = convert_from_path(pdf_path, dpi=self.dpi)

            for i, page in enumerate(pages, start=1):
                img_name = f"page_{i}.png"
                img_path = os.path.join(folder, img_name)
                page.save(img_path, "PNG")
                logger.info(f" Saved {img_path}")

                # OCR the page to detect "closed deal sheet"
                try:
                    text = pytesseract.image_to_string(page).lower()
                except Exception as e:
                    logger.warning(f"OCR failed for {img_path}: {e}")
                    text = ""

                if "closed deal sheet" in text:
                    closed_name = f"{base}_page_{i}.png"
                    closed_path = os.path.join(self.closed_dir, closed_name)
                    page.save(closed_path, "PNG")
                    logger.info(f" → Detected 'closed deal sheet'; saved to {closed_path}")


if __name__ == "__main__":
    converter = PDFToImageConverter(
        pdf_dir="/dbfs/mnt/mini-proj-dd/contract_docs",
        output_dir="/dbfs/mnt/mini-proj-dd/contract_docs"
    )
    converter.convert_all()


2025-04-22 15:22:34,449 [INFO] Found 5 PDFs in /dbfs/mnt/mini-proj-dd/contract_docs
2025-04-22 15:22:34,458 [INFO] Converting 1.pdf → images in /dbfs/mnt/mini-proj-dd/contract_docs/images/1
2025-04-22 15:22:40,597 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_1.png
2025-04-22 15:22:45,341 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_2.png
2025-04-22 15:22:49,976 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_3.png
2025-04-22 15:22:55,556 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_4.png
2025-04-22 15:22:58,769 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_5.png
2025-04-22 15:23:02,051 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_6.png
2025-04-22 15:23:04,841 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_7.png
2025-04-22 15:23:07,562 [INFO]  → Detected 'closed deal sheet'; saved to /dbfs/mnt/mini-proj-dd/contract_docs/images/closed_deal_sheets/1_page_7.

This script is meant to be used as a cluster init script so that dependencies are installed automatically when the cluster starts. 

Use Tesseract OCR for extracting text from image or PDF documents.

Use Poppler (pdftoppm) to convert PDFs to images before OCR.

In [0]:
#DND

import logging

# 1) Configure root logger to INFO (so nothing below INFO shows)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# 2) Silence noisy back‑end loggers completely
for name in (
    "py4j",
    "py4j.java_gateway",
    "pyspark",
    "databricks",
    "dbruntime",
    "jedi",
    "pdf2image"
):
    logging.getLogger(name).setLevel(logging.WARNING)

# 3) Create your converter logger with exactly one handler
logger = logging.getLogger("PDFConverter")
logger.setLevel(logging.INFO)
logger.propagate = False       # don’t bubble up to root

# Drop any old handlers (e.g. from reruns)
for h in list(logger.handlers):
    logger.removeHandler(h)

# Add a single StreamHandler
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(ch)


In [0]:
#dnd
class PDFToImageConverter:
    """
    Converts all PDFs in `pdf_dir` into per‑page PNGs under
    `output_dir/<pdf_basename>/page_<n>.png`.
    """

    def __init__(self, pdf_dir: str, output_dir: str, dpi: int = 300):
        """
        :param pdf_dir:    e.g. "/dbfs/mnt/mini-proj-dd/contract_docs"
        :param output_dir: e.g. "/dbfs/mnt/mini-proj-dd/contract_docs/images"
        :param dpi:        resolution for conversion
        """
        self.pdf_dir = pdf_dir
        self.output_dir = output_dir
        self.dpi = dpi
        os.makedirs(self.output_dir, exist_ok=True)

    def convert_all(self):
        """
        For each PDF in pdf_dir:
          - create folder output_dir/<basename>
          - convert pages to PNG
          - save as page_1.png, page_2.png, etc.
        """
        pdf_files = [
            f for f in os.listdir(self.pdf_dir)
            if f.lower().endswith(".pdf")
        ]
        logger.info(f"Found {len(pdf_files)} PDFs in {self.pdf_dir}")

        for pdf in pdf_files:
            base = os.path.splitext(pdf)[0]
            folder = os.path.join(self.output_dir, base)
            os.makedirs(folder, exist_ok=True)

            logger.info(f"Converting {pdf} → images in {folder}")
            pdf_path = os.path.join(self.pdf_dir, pdf)
            pages = convert_from_path(pdf_path, dpi=self.dpi)

            for i, page in enumerate(pages, start=1):
                img_path = os.path.join(folder, f"page_{i}.png")
                page.save(img_path, "PNG")
                logger.info(f" Saved {img_path}")


# ──────────────────────────────────────────────────────────────────────────────
# Usage (run once)
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    CONVERTER = PDFToImageConverter(
        pdf_dir="/dbfs/mnt/mini-proj-dd/contract_docs",
        output_dir="/dbfs/mnt/mini-proj-dd/contract_docs/images"
    )
    CONVERTER.convert_all()


2025-04-21 19:35:07,440 [INFO] Found 5 PDFs in /dbfs/mnt/mini-proj-dd/contract_docs
2025-04-21 19:35:07,445 [INFO] Converting 1.pdf → images in /dbfs/mnt/mini-proj-dd/contract_docs/images/1
2025-04-21 19:35:34,717 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_1.png
2025-04-21 19:35:35,572 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_2.png
2025-04-21 19:35:36,570 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_3.png
2025-04-21 19:35:37,424 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_4.png
2025-04-21 19:35:38,225 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_5.png
2025-04-21 19:35:38,978 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_6.png
2025-04-21 19:35:39,710 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_7.png
2025-04-21 19:35:40,730 [INFO]  Saved /dbfs/mnt/mini-proj-dd/contract_docs/images/1/page_8.png
2025-04-21 19:35:41,549 [INFO]  Saved /dbfs/mnt/mi

Reading package lists...


E: Could not get lock /var/lib/apt/lists/lock. It is held by process 1870 (apt-get)
E: Unable to lock directory /var/lib/apt/lists/


path,name,size,modificationTime
dbfs:/databricks/init/install_poppler.sh,install_poppler.sh,142,1743112021000
dbfs:/databricks/init/install_system_dependencies.sh,install_system_dependencies.sh,203,1745335142000


Wrote 203 bytes.


...

'/dbfs/models/t5-small'

▶️ Downloading Donut into /dbfs/models/donut-base …


/databricks/python/lib/python3.12/site-packages/huggingface_hub/file_download.py:832: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

✅ Donut snapshot complete
